In [2]:
import pandas as pd

import os
os.makedirs("Results", exist_ok=True)
# Load datasets
balls = pd.read_csv(r"C:\Users\Admin\Downloads\New folder\ball_by_ball.csv")
matches = pd.read_csv(r"C:\Users\Admin\Downloads\New folder\matches.csv")

# Keep normal innings only
balls = balls[balls["innings"].isin([1, 2])].copy()

# Calculate total runs
balls["total_runs"] = (
    balls["runs"] +
    balls["extras1_runs"] +
    balls["extras_2_runs"]
)

# Identify legal deliveries
balls["legal_ball"] = ~balls["extras_type"].isin(
    ["wide", "Overstep"]
)

# Create innings summary
summary = balls.groupby(
    ["match_id", "innings", "batting_team_id"]
).agg(
    total_runs=("total_runs", "sum"),
    wickets=("is_wicket", "sum"),
    legal_balls=("legal_ball", "sum")
).reset_index()

# Calculate overs
summary["overs"] = (
    summary["legal_balls"] // 6
    + (summary["legal_balls"] % 6) / 10
)

# Add match information
summary = summary.merge(
    matches[[
        "match_id", "match_date", "stage",
        "home_team", "away_team",
        "revised_target"
    ]],
    on="match_id",
    how="left"
)

print("\n===== INNINGS SUMMARY =====")
print(summary.head(15).to_string(index=False))

# Save results
summary.to_csv("Results/innings_summary.csv", index=False)

print("\nInnings summary created successfully!")



===== INNINGS SUMMARY =====
 match_id  innings  batting_team_id  total_runs  wickets  legal_balls  overs match_date  stage   home_team   away_team  revised_target
    13511        1               22         103       10          118   19.4 2022-01-22 League West Indies     England             NaN
    13511        2               28         104        1          103   17.1 2022-01-22 League West Indies     England             NaN
    13512        1               22         171        8          120   20.0 2022-01-23 League West Indies     England             NaN
    13512        2               28         170        8          120   20.0 2022-01-23 League West Indies     England             NaN
    13513        1               28         224        5          121   20.1 2022-01-26 League West Indies     England             NaN
    13513        2               22         204        9          120   20.0 2022-01-26 League West Indies     England             NaN
    13514        1        